# 02. Data Cleaning & Preprocessing
## Intelligent Cyber Threat Detection & Network Traffic Analysis
**Benchmark Dataset:** CIC-IDS2017

---
### 1. Objectives & Anti-Leakage Methodology
This notebook implements the data sanitization pipeline:
1. Converting infinite values to `NaN`.
2. Median imputation of missing numerical values.
3. Elimination of duplicate flows.
4. Identification and dropping of constant features (variance = 0).
5. Removal of leakage-prone identifier fields (IPs, MACs, Flow IDs, Timestamps).
6. Robust feature scaling (`StandardScaler`) fitted **strictly on training partitions**.

In [ ]:
import os
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import pandas as pd
import numpy as np
from src.data_loader import load_dataset
from src.preprocessing import NetworkDataPreprocessor, clean_labels, prepare_train_test_split

df, _ = load_dataset(use_sample_if_empty=True)
print(f"Loaded initial dataset: {df.shape}")

### 2. Demonstrating Anti-Leakage Preprocessing
Splitting records into stratified training and testing sets before fitting transformations.

In [ ]:
X_train, X_test, y_train, y_test, preprocessor, meta = prepare_train_test_split(
    df, mode="binary", test_size=0.25, random_state=42
)

print(f"X_train scaled shape: {X_train.shape}")
print(f"X_test scaled shape: {X_test.shape}")
print(f"Dropped constant features: {preprocessor.dropped_constant_features_}")
print(f"Active features retained: {len(preprocessor.feature_names_)}")
print(f"Target class distribution: {meta['classes']}")
print(f"Calculated class weights: {meta['class_weights']}")

### 3. Verification of Cleaned Arrays
Verifying that no NaNs or Infs persist in transformed arrays.

In [ ]:
print("Verification Checks:")
print(f"  Any NaN in X_train? {np.isnan(X_train).any()}")
print(f"  Any Inf in X_train? {np.isinf(X_train).any()}")
print(f"  Any NaN in X_test? {np.isnan(X_test).any()}")
print(f"  Any Inf in X_test? {np.isinf(X_test).any()}")
print(f"  Feature mean range (scaled): [{X_train.mean(axis=0).min():.2f}, {X_train.mean(axis=0).max():.2f}]")
print(f"  Feature std range (scaled): [{X_train.std(axis=0).min():.2f}, {X_train.std(axis=0).max():.2f}]")

### 4. Summary & Verification Conclusion
- All data cleaning parameters (medians, scaling means, std deviations, clip boundaries) are computed exclusively on `X_train`.
- Test data `X_test` is transformed without recalculating parameters, guaranteeing zero train/test contamination.